BOND

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pymatgen.core import Structure
from pymatgen.analysis.local_env import CrystalNN
from scipy.optimize import linear_sum_assignment

def find_motifs(contcar_file, magnetic_species, num_neighbors):
    """
    Extract the two magnetic-atom-centred motifs from a CONTCAR file.
    
    Each motif contains:
      - center_coord: coordinates of the central magnetic atom
      - motif_coords: positions of the selected ligands relative to the central atom (centred coordinates)
      - selected_neighbor_indices: original indices of the selected ligands
      - distances: distance from the centre to each neighbour
      
    Parameters:
      contcar_file (str): path to the CONTCAR file
      magnetic_species (list): list of magnetic element symbols
      num_neighbors (int): maximum number of ligands to select around each magnetic atom
      
    Returns:
      motifs (dict): keyed by the two magnetic-atom indices, holding each motif
    """
    structure = Structure.from_file(contcar_file)
    
    # find the site indices of the magnetic atoms
    magnetic_indices = [i for i, site in enumerate(structure) if str(site.specie) in magnetic_species]
    if len(magnetic_indices) < 2:
        raise ValueError("fewer than two magnetic atoms in the CONTCAR")
    
    # if more than two magnetic atoms are present, use only the first two
    magnetic_indices = magnetic_indices[:2]
    
    cnn = CrystalNN()
    motifs = {}
    
    for mag_index in magnetic_indices:
        center_site = structure[mag_index]
        nn_info = cnn.get_nn_info(structure, mag_index)

        # keep only ligands (elements absent from the magnetic-species list)
        nonmag_neighbors = []
        nonmag_neighbor_indices = []
        for info in nn_info:
            neighbor_index = info.get("site_index", None)
            neighbor_site = info["site"]
            if str(neighbor_site.specie) not in magnetic_species:
                nonmag_neighbors.append(neighbor_site.coords)
                nonmag_neighbor_indices.append(neighbor_index)
        
        nonmag_neighbors = np.array(nonmag_neighbors)
        
        if nonmag_neighbors.shape[0] == 0:
            raise ValueError(f"no ligand found around magnetic atom index {mag_index}")
        
        # select the nearest neighbours by distance from the centre
        distances = np.linalg.norm(nonmag_neighbors - center_site.coords, axis=1)
        actual_neighbors = min(num_neighbors, nonmag_neighbors.shape[0])
        sorted_idx = np.argsort(distances)[:actual_neighbors]
        selected_neighbors = nonmag_neighbors[sorted_idx]
        selected_indices = [nonmag_neighbor_indices[i] for i in sorted_idx]
        selected_distances = distances[sorted_idx]
        
        # subtract the centre to obtain centred relative coordinates
        motif_coords = selected_neighbors - center_site.coords
        
        motifs[mag_index] = {
            "center_coord": center_site.coords,
            "motif_coords": motif_coords,
            "selected_neighbor_indices": selected_indices,
            "distances": selected_distances
        }
    
    return motifs

def sort_motif_by_neighbor_indices(motif):
    """
    Sort the motif dict in ascending order of 'selected_neighbor_indices' and
    reorder 'motif_coords' and 'distances' to match.
    """
    sel_idx = np.array(motif["selected_neighbor_indices"])
    order = np.argsort(sel_idx)
    
    motif["selected_neighbor_indices"] = sel_idx[order].tolist()
    motif["motif_coords"] = motif["motif_coords"][order]
    motif["distances"] = motif["distances"][order]
    return motif

def compute_bond_length_stats(motif):
    """
    For the bond lengths within the motif (magnetic atom to ligand), 
    return the mean, maximum, minimum and standard deviation as a dict.
    """
    distances = motif["distances"]
    stats = {
        "avg_bond_length": np.mean(distances),
        "max_bond_length": np.max(distances),
        "min_bond_length": np.min(distances),
        "std_bond_length": np.std(distances)
    }
    return stats

def compute_magnetic_center_angles(motif):
    """
    Ligand-metal-ligand angles: the angle between two ligand vectors at the centre (the magnetic atom, at the origin).
    
    The angle is computed for every pair of points in motif["motif_coords"].
    (angles in degrees)
    
    Returns:
      stats (dict): maximum, minimum, mean and standard deviation of the angles.
    """
    points = motif["motif_coords"]
    n = points.shape[0]
    angles = []
    if n < 2:
        return {"max_angle": None, "min_angle": None, "avg_angle": None, "std_angle": None}
    
    for i in range(n):
        for j in range(i+1, n):
            v1 = points[i]
            v2 = points[j]
            norm1 = np.linalg.norm(v1)
            norm2 = np.linalg.norm(v2)
            if norm1 == 0 or norm2 == 0:
                continue
            cosine_angle = np.dot(v1, v2) / (norm1 * norm2)
            cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
            angle = np.arccos(cosine_angle) * (180.0 / np.pi)
            angles.append(angle)
    angles = np.array(angles)
    if angles.size == 0:
        return {"max_angle": None, "min_angle": None, "avg_angle": None, "std_angle": None}
    stats = {
        "max_angle": np.max(angles),
        "min_angle": np.min(angles),
        "avg_angle": np.mean(angles),
        "std_angle": np.std(angles)
    }
    return stats

def compute_nonmagnetic_vertex_angles(motif):
    """
    Ligand-ligand-ligand angles: taking one ligand of the motif as the vertex,
    the angle it subtends with two other ligands is computed.
    
    That is, for three points A, B and C in the motif, angle(ABC) is taken with B as the vertex.
    
    Returns:
      stats (dict): maximum, minimum, mean and standard deviation of the angles, in degrees.
    """
    points = motif["motif_coords"]
    n = points.shape[0]
    angles = []
    if n < 3:
        return {"max_angle": None, "min_angle": None, "avg_angle": None, "std_angle": None}
    
    for vertex in range(n):
        others = [i for i in range(n) if i != vertex]
        for i in range(len(others)):
            for j in range(i+1, len(others)):
                p1 = points[others[i]]
                p2 = points[others[j]]
                vertex_point = points[vertex]
                v1 = p1 - vertex_point
                v2 = p2 - vertex_point
                norm1 = np.linalg.norm(v1)
                norm2 = np.linalg.norm(v2)
                if norm1 == 0 or norm2 == 0:
                    continue
                cosine_angle = np.dot(v1, v2) / (norm1 * norm2)
                cosine_angle = np.clip(cosine_angle, -1.0, 1.0)
                angle = np.arccos(cosine_angle) * (180.0 / np.pi)
                angles.append(angle)
    angles = np.array(angles)
    if angles.size == 0:
        return {"max_angle": None, "min_angle": None, "avg_angle": None, "std_angle": None}
    stats = {
        "max_angle": np.max(angles),
        "min_angle": np.min(angles),
        "avg_angle": np.mean(angles),
        "std_angle": np.std(angles)
    }
    return stats

# build the DataFrame by collecting the per-file dicts
data_list = []
# use the CONTCAR pattern rather than POSCAR (adjust for your setup)
contcar_files = glob.glob(os.path.join("", "POSCAR_*"))
file_list = [f.replace(os.sep, "/") for f in contcar_files]
magnetic_species = ["Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", 
                          "Y", "Zr", "Nb", "Mo", "Ru", "Rh", "Pd", "Ag", "Cd"]
for contcar_file in file_list:
    try:
        motifs = find_motifs(contcar_file, magnetic_species=magnetic_species, num_neighbors=12)
    except Exception:
        continue  # skip on error
    
    keys = list(motifs.keys())
    if len(keys) < 2:
        continue  # skip if fewer than two magnetic atoms
    
    motif0 = sort_motif_by_neighbor_indices(motifs[keys[0]])
    motif1 = sort_motif_by_neighbor_indices(motifs[keys[1]])
    
    bond_stats0 = compute_bond_length_stats(motif0)
    bond_stats1 = compute_bond_length_stats(motif1)
    combined_bond_stats = {
        "avg_bond_length": (bond_stats0["avg_bond_length"] + bond_stats1["avg_bond_length"]) / 2,
        "max_bond_length": (bond_stats0["max_bond_length"] + bond_stats1["max_bond_length"]) / 2,
        "min_bond_length": (bond_stats0["min_bond_length"] + bond_stats1["min_bond_length"]) / 2,
        "std_bond_length": (bond_stats0["std_bond_length"] + bond_stats1["std_bond_length"]) / 2,
    }
    
    center_angles0 = compute_magnetic_center_angles(motif0)
    center_angles1 = compute_magnetic_center_angles(motif1)
    combined_center_angles = {
        "center_max_angle": (center_angles0["max_angle"] + center_angles1["max_angle"]) / 2 if center_angles0["max_angle"] is not None and center_angles1["max_angle"] is not None else None,
        "center_min_angle": (center_angles0["min_angle"] + center_angles1["min_angle"]) / 2 if center_angles0["min_angle"] is not None and center_angles1["min_angle"] is not None else None,
        "center_avg_angle": (center_angles0["avg_angle"] + center_angles1["avg_angle"]) / 2 if center_angles0["avg_angle"] is not None and center_angles1["avg_angle"] is not None else None,
        "center_std_angle": (center_angles0["std_angle"] + center_angles1["std_angle"]) / 2 if center_angles0["std_angle"] is not None and center_angles1["std_angle"] is not None else None,
    }
    
    nonmag_angles0 = compute_nonmagnetic_vertex_angles(motif0)
    nonmag_angles1 = compute_nonmagnetic_vertex_angles(motif1)
    combined_nonmag_angles = {
        "nonmag_max_angle": (nonmag_angles0["max_angle"] + nonmag_angles1["max_angle"]) / 2 if nonmag_angles0["max_angle"] is not None and nonmag_angles1["max_angle"] is not None else None,
        "nonmag_min_angle": (nonmag_angles0["min_angle"] + nonmag_angles1["min_angle"]) / 2 if nonmag_angles0["min_angle"] is not None and nonmag_angles1["min_angle"] is not None else None,
        #"nonmag_avg_angle": (nonmag_angles0["avg_angle"] + nonmag_angles1["avg_angle"]) / 2 if nonmag_angles0["avg_angle"] is not None and nonmag_angles1["avg_angle"] is not None else None,
        "nonmag_std_angle": (nonmag_angles0["std_angle"] + nonmag_angles1["std_angle"]) / 2 if nonmag_angles0["std_angle"] is not None and nonmag_angles1["std_angle"] is not None else None,
    }
    
    file_data = {"filename": contcar_file}
    file_data.update(combined_bond_stats)
    file_data.update(combined_center_angles)
    file_data.update(combined_nonmag_angles)
    data_list.append(file_data)

# build the DataFrame (stored in df, not printed)
BOND_df = pd.DataFrame(data_list)
BOND_df

In [ ]:
rows_with_nan = BOND_df[BOND_df.isna().any(axis=1)]
rows_with_nan

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pymatgen.core import Structure

def distinct_values(sorted_values, threshold=0.01):
    """
    In a sorted list of values, consecutive values differing by at most the threshold are treated as equal,
    and only the distinct values are returned.
    """
    distinct = []
    for d in sorted_values:
        if not distinct or (d - distinct[-1]) > threshold:
            distinct.append(d)
    return distinct

def compute_labelled_min_magnetic_distances(structure, magnetic_species, num_neighbors=3, translation_range=(-1, 0, 1), threshold=0.1):
    """
    Assuming the structure has exactly two magnetic atoms, 
    the periodic-image distances are collected in both directions, Ag1->Ag2 and Ag2->Ag1.
    At each rank the smaller of the two directional values is used.
    
    Returns:
      features (dict): holding "labelled_1st", "labelled_2nd" and "labelled_3rd".
    """
    # magnetic-atom indices (exactly two)
    magnetic_indices = [i for i, site in enumerate(structure) if str(site.specie) in magnetic_species]
    if len(magnetic_indices) != 2:
        raise ValueError("the structure must contain exactly two magnetic atoms")
    
    idx1, idx2 = magnetic_indices
    lattice = structure.lattice
    coords1 = structure[idx1].coords
    coords2 = structure[idx2].coords
    
    forward = []
    reverse = []
    for i in translation_range:
        for j in translation_range:
            for k in translation_range:
                t = i * lattice.matrix[0] + j * lattice.matrix[1] + k * lattice.matrix[2]
                d_f = np.linalg.norm(coords2 + t - coords1)
                d_r = np.linalg.norm(coords1 + t - coords2)
                forward.append(d_f)
                reverse.append(d_r)
    forward = np.sort(np.array(forward))
    reverse = np.sort(np.array(reverse))
    
    distinct_forward = distinct_values(forward, threshold)
    distinct_reverse = distinct_values(reverse, threshold)
    
    labelled = []
    for i in range(num_neighbors):
        d1 = distinct_forward[i] if i < len(distinct_forward) else np.nan
        d2 = distinct_reverse[i] if i < len(distinct_reverse) else np.nan
        # take the smaller of the two where neither is nan
        if np.isnan(d1) and np.isnan(d2):
            labelled.append(np.nan)
        elif np.isnan(d1):
            labelled.append(d2)
        elif np.isnan(d2):
            labelled.append(d1)
        else:
            labelled.append(min(d1, d2))
    
    features = {
        "labelled_1st": labelled[0],
        "labelled_2nd": labelled[1],
        "labelled_3rd": labelled[2]
    }
    return features

def compute_global_min_magnetic_distances(structure, magnetic_species, num_neighbors=3, translation_range=(-1, 0, 1), threshold=0.1):
    """
    When the structure has exactly two magnetic atoms,  
    (1) all cross periodic-image distances between Ag1 and Ag2, and  
    (2) the non-trivial periodic self-image distances of each atom (excluding the trivial (0,0,0))
    are computed and combined.
    The distinct values are then extracted and the smallest at each rank is used.
    
    Returns:
      features (dict): holding "global_1st", "global_2nd" and "global_3rd".
    """
    magnetic_indices = [i for i, site in enumerate(structure) if str(site.specie) in magnetic_species]
    if len(magnetic_indices) != 2:
        raise ValueError("the structure must contain exactly two magnetic atoms")
    
    lattice = structure.lattice
    coords = [structure[i].coords for i in magnetic_indices]
    
    all_distances = []
    # cross distances: Ag1->Ag2, both directions
    for i in range(2):
        for j in range(2):
            if i != j:
                for a in translation_range:
                    for b in translation_range:
                        for c in translation_range:
                            t = a * lattice.matrix[0] + b * lattice.matrix[1] + c * lattice.matrix[2]
                            d = np.linalg.norm(coords[j] + t - coords[i])
                            all_distances.append(d)
    # self distances, excluding (0,0,0) for each atom
    for i in range(2):
        for a in translation_range:
            for b in translation_range:
                for c in translation_range:
                    if a == 0 and b == 0 and c == 0:
                        continue
                    t = a * lattice.matrix[0] + b * lattice.matrix[1] + c * lattice.matrix[2]
                    d = np.linalg.norm(coords[i] + t - coords[i])
                    all_distances.append(d)
    all_distances = np.array(all_distances)
    all_distances = np.sort(all_distances)
    distinct_all = distinct_values(all_distances, threshold)
    
    features = {}
    features["global_1st"] = distinct_all[0] if len(distinct_all) >= 1 else np.nan
    features["global_2nd"] = distinct_all[1] if len(distinct_all) >= 2 else np.nan
    features["global_3rd"] = distinct_all[2] if len(distinct_all) >= 3 else np.nan
    
    return features

# collect the data and build the DataFrame
data_list = []
# CONTCAR files are used here
contcar_files = glob.glob(os.path.join("", "POSCAR_*"))
file_list = [f.replace(os.sep, "/") for f in contcar_files]

# magnetic elements to use (example)
magnetic_species = ["Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", 
                    "Y", "Zr", "Nb", "Mo", "Ru", "Rh", "Pd", "Ag", "Cd"]

for contcar_file in file_list:
    try:
        structure = Structure.from_file(contcar_file)
    except Exception:
        continue
    try:
        labelled_features = compute_labelled_min_magnetic_distances(
            structure, magnetic_species, num_neighbors=3, translation_range=(-1, 0, 1), threshold=0.1)
        global_features = compute_global_min_magnetic_distances(
            structure, magnetic_species, num_neighbors=3, translation_range=(-1, 0, 1), threshold=0.1)
    except Exception:
        continue
    
    file_data = {"filename": contcar_file}
    file_data.update(labelled_features)
    file_data.update(global_features)
    data_list.append(file_data)

# build the DataFrame (stored in df)
NN_df = pd.DataFrame(data_list)
NN_df

In [ ]:
rows_with_nan = NN_df[NN_df.isna().any(axis=1)]
rows_with_nan

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pymatgen.core import Structure
from pymatgen.analysis.local_env import CrystalNN
from scipy.spatial import ConvexHull, QhullError

def find_motifs(contcar_file, magnetic_species, num_neighbors):
    structure = Structure.from_file(contcar_file)
    magnetic_indices = [i for i, site in enumerate(structure) if str(site.specie) in magnetic_species]
    if len(magnetic_indices) < 2:
        raise ValueError("fewer than two magnetic atoms in the CONTCAR")
    magnetic_indices = magnetic_indices[:2]

    cnn = CrystalNN()
    motifs = {}
    for mag_index in magnetic_indices:
        center_site = structure[mag_index]
        nn_info = cnn.get_nn_info(structure, mag_index)
        nonmag_neighbors = []
        for info in nn_info:
            neighbor_site = info["site"]
            if str(neighbor_site.specie) not in magnetic_species:
                nonmag_neighbors.append(neighbor_site.coords)
        nonmag_neighbors = np.array(nonmag_neighbors)
        if nonmag_neighbors.shape[0] == 0:
            raise ValueError(f"no ligand found around magnetic atom index {mag_index}")
        distances = np.linalg.norm(nonmag_neighbors - center_site.coords, axis=1)
        actual_neighbors = min(num_neighbors, nonmag_neighbors.shape[0])
        sorted_idx = np.argsort(distances)[:actual_neighbors]
        selected_neighbors = nonmag_neighbors[sorted_idx]
        motif_coords = selected_neighbors - center_site.coords
        motifs[mag_index] = {"center_coord": center_site.coords, "motif_coords": motif_coords}
    return motifs

def compute_motif_properties_2d(points, tolerance=1e-6):
    """Convex-hull area and plane normal of a 2D motif."""
    if points.shape[0] < 3:
        return None, None, None
    
    center = np.mean(points, axis=0)
    centered = points - center
    
    try:
        U, S, Vt = np.linalg.svd(centered, full_matrices=False)
    except np.linalg.LinAlgError:
        return None, None, None
    
    if len(S) < 2 or S[1] < tolerance:
        return None, None, None
    
    projection_matrix = Vt[:2, :]
    projected = np.dot(centered, projection_matrix.T)
    
    # Plane normal vector
    plane_normal = Vt[2, :]
    plane_normal = plane_normal / np.linalg.norm(plane_normal)
    
    try:
        hull2d = ConvexHull(projected)
        area = hull2d.volume
    except (QhullError, ValueError):
        return None, None, None
    
    return area, area, plane_normal

def compute_motif_properties_3d(points):
    """Convex-hull volume of a 3D motif."""
    if points.shape[0] < 4:
        return None, None
    
    try:
        hull = ConvexHull(points)
        return hull.volume, hull.area
    except (QhullError, ValueError):
        return None, None

def compute_motif_convex_properties(motif, tolerance=1e-6):
    """Determine the motif dimensionality and compute the corresponding convex hull."""
    points = motif["motif_coords"]
    if points.shape[0] < 3:
        return {"hull_volume": None, "hull_area": None, "dimension": None, "plane_normal": None}
    
    centered = points - np.mean(points, axis=0)
    
    try:
        U, S, Vt = np.linalg.svd(centered, full_matrices=False)
        rank = np.sum(S > tolerance * S[0]) if S[0] > 0 else 0
    except np.linalg.LinAlgError:
        return {"hull_volume": None, "hull_area": None, "dimension": None, "plane_normal": None}
    
    if rank == 2:
        hull_area, _, plane_normal = compute_motif_properties_2d(points, tolerance)
        return {
            "hull_volume": None,
            "hull_area": hull_area,
            "dimension": 2,
            "plane_normal": plane_normal
        }
    elif rank >= 3:
        hull_vol, hull_area = compute_motif_properties_3d(points)
        return {
            "hull_volume": hull_vol,
            "hull_area": hull_area,
            "dimension": 3,
            "plane_normal": None
        }
    else:
        return {"hull_volume": None, "hull_area": None, "dimension": rank, "plane_normal": None}

def find_most_parallel_face(structure, motif_plane_normal):
    """Find the unit-cell face most parallel to the motif plane."""
    lattice = structure.lattice
    a = lattice.matrix[0]
    b = lattice.matrix[1]
    c = lattice.matrix[2]
    
    # normal vector of each face
    normal_ab = np.cross(a, b)
    normal_bc = np.cross(b, c)
    normal_ca = np.cross(c, a)
    
    # normalise
    normal_ab = normal_ab / np.linalg.norm(normal_ab)
    normal_bc = normal_bc / np.linalg.norm(normal_bc)
    normal_ca = normal_ca / np.linalg.norm(normal_ca)
    
    # compute the angle
    cos_angle_ab = abs(np.dot(motif_plane_normal, normal_ab))
    cos_angle_bc = abs(np.dot(motif_plane_normal, normal_bc))
    cos_angle_ca = abs(np.dot(motif_plane_normal, normal_ca))
    
    angle_ab = np.arccos(np.clip(cos_angle_ab, 0, 1)) * 180 / np.pi
    angle_bc = np.arccos(np.clip(cos_angle_bc, 0, 1)) * 180 / np.pi
    angle_ca = np.arccos(np.clip(cos_angle_ca, 0, 1)) * 180 / np.pi
    
    # compute the area
    area_ab = np.linalg.norm(np.cross(a, b))
    area_bc = np.linalg.norm(np.cross(b, c))
    area_ca = np.linalg.norm(np.cross(c, a))
    
    angles = [angle_ab, angle_bc, angle_ca]
    areas = [area_ab, area_bc, area_ca]
    face_types = ['ab', 'bc', 'ca']
    
    best_idx = np.argmin(angles)
    
    return {
        'best_face_area': areas[best_idx],
        'best_face_type': face_types[best_idx],
        'angle_degrees': angles[best_idx],
        'median_face_area': np.median(areas),
        'all_areas': areas,
        'all_angles': angles
    }

def compute_unit_cell_measure_2d_hybrid(structure, motif_plane_normal, safety_factor=0.5):
    """
    ⭐ HYBRID METHOD ⭐
    
    Use the most parallel face by default, with a safeguard:
    - if the parallel face is smaller than safety_factor times the median, use the median instead
    - this minimises the risk of a packing fraction above 1
    
    Parameters:
    -----------
    structure : Structure
    motif_plane_normal : ndarray
        normal vector of the motif plane
    safety_factor : float
        safety threshold (default 0.5)
        if the parallel face is below safety_factor times the median, the median is used
    
    Returns:
    --------
    dict with:
        'area': the area actually used
        'method': 'parallel_face' or 'median_safeguard'
        'parallel_area': area of the parallel face
        'median_area': the median area
        'safety_triggered': whether the safeguard fired
    """
    face_info = find_most_parallel_face(structure, motif_plane_normal)
    parallel_area = face_info['best_face_area']
    median_area = face_info['median_face_area']
    
    # check the safeguard
    if parallel_area < safety_factor * median_area:
        # the parallel face is too small -> use the median
        selected_area = median_area
        method = "median_safeguard"
        safety_triggered = True
    else:
        # the parallel face is large enough -> use it
        selected_area = parallel_area
        method = "parallel_face"
        safety_triggered = False
    
    return {
        'area': selected_area,
        'method': method,
        'parallel_area': parallel_area,
        'median_area': median_area,
        'safety_triggered': safety_triggered,
        'best_face_type': face_info['best_face_type'],
        'angle_degrees': face_info['angle_degrees']
    }

def compute_comparable_metrics(raw_ratio, dimension):
    """Convert 2D and 3D quantities to a comparable measure."""
    if raw_ratio is None or dimension is None:
        return {
            "characteristic_length_ratio": None,
            "packing_fraction": None,
        }
    
    if dimension == 2:
        characteristic_length_ratio = np.sqrt(raw_ratio)
    elif dimension == 3:
        characteristic_length_ratio = np.cbrt(raw_ratio)
    else:
        characteristic_length_ratio = raw_ratio
    
    return {
        "characteristic_length_ratio": characteristic_length_ratio,
        "packing_fraction": raw_ratio,
    }

# list to collect the results
data_list = []

# list of CONTCAR files
contcar_files = glob.glob(os.path.join("", "POSCAR_*"))
file_list = [f.replace(os.sep, "/") for f in contcar_files]

# settings
SAFETY_FACTOR = 0.5  # adjustable; 0.3-0.7 recommended

print(f"Found {len(file_list)} POSCAR files to process")
print("\n" + "="*80)
print("⭐ HYBRID METHOD ⭐")
print("  Strategy: Most parallel face by default")
print("  Safety: Use median if parallel face < 50% of median")
print(f"  Safety factor: {SAFETY_FACTOR}")
print("  Goal: Maximize accuracy while minimizing f>1 risk")
print("="*80 + "\n")

safety_triggered_count = 0

for idx, contcar_file in enumerate(file_list, 1):
    if idx % 10 == 0:
        print(f"Processing {idx}/{len(file_list)}...")
    
    try:
        motifs = find_motifs(contcar_file, 
                             magnetic_species=["Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", 
                                               "Y", "Zr", "Nb", "Mo", "Ru", "Rh", "Pd", "Ag", "Cd"], 
                             num_neighbors=12)
    except Exception as e:
        continue
    
    keys = list(motifs.keys())
    if len(keys) < 2:
        continue
    
    motif0 = find_motifs.__wrapped__ if hasattr(find_motifs, '__wrapped__') else motifs[keys[0]]
    motif1 = motifs[keys[1]]
    
    motif0 = motifs[keys[0]]
    motif1 = motifs[keys[1]]
    
    # convex-hull properties of each motif
    convex0 = compute_motif_convex_properties(motif0)
    convex1 = compute_motif_convex_properties(motif1)
    
    # skip if the two motifs have different dimensionality
    if convex0["dimension"] != convex1["dimension"]:
        continue
    
    if convex0["dimension"] is None or convex0["dimension"] < 2:
        continue
    
    dimension = convex0["dimension"]
    
    # structure information
    structure = Structure.from_file(contcar_file)
    unit_cell_volume = structure.lattice.volume
    
    # the calculation differs by dimensionality
    if dimension == 2:
        # 2D motif: HYBRID METHOD
        if (convex0["hull_area"] is not None and convex1["hull_area"] is not None and
            convex0["plane_normal"] is not None and convex1["plane_normal"] is not None):
            
            avg_motif_measure = (convex0["hull_area"] + convex1["hull_area"]) / 2
            
            # mean plane normal
            avg_plane_normal = (convex0["plane_normal"] + convex1["plane_normal"]) / 2
            avg_plane_normal = avg_plane_normal / np.linalg.norm(avg_plane_normal)
            
            # apply the hybrid method
            cell_info = compute_unit_cell_measure_2d_hybrid(
                structure, avg_plane_normal, safety_factor=SAFETY_FACTOR
            )
            
            unit_cell_measure = cell_info['area']
            method_used = cell_info['method']
            
            if cell_info['safety_triggered']:
                safety_triggered_count += 1
            
            if unit_cell_measure > 0:
                ratio = avg_motif_measure / unit_cell_measure
            else:
                ratio = None
            
            # additional information
            best_face_type = cell_info['best_face_type']
            parallel_angle = cell_info['angle_degrees']
            parallel_area = cell_info['parallel_area']
            median_area = cell_info['median_area']
            safety_triggered = cell_info['safety_triggered']
            
        else:
            avg_motif_measure = None
            unit_cell_measure = None
            ratio = None
            method_used = None
            best_face_type = None
            parallel_angle = None
            parallel_area = None
            median_area = None
            safety_triggered = None
            
    elif dimension == 3:
        # 3D motif
        if convex0["hull_volume"] is not None and convex1["hull_volume"] is not None:
            avg_motif_measure = (convex0["hull_volume"] + convex1["hull_volume"]) / 2
            unit_cell_measure = unit_cell_volume
            
            if unit_cell_measure > 0:
                ratio = avg_motif_measure / unit_cell_measure
            else:
                ratio = None
        else:
            avg_motif_measure = None
            unit_cell_measure = unit_cell_volume
            ratio = None
        
        method_used = "volume_ratio"
        best_face_type = None
        parallel_angle = None
        parallel_area = None
        median_area = None
        safety_triggered = None
    else:
        continue
    
    # warn if the value is still above 1
    if ratio is not None and ratio > 1.0:
        print(f"⚠️  WARNING: {contcar_file}")
        print(f"  f = {ratio:.3f} > 1.0 even with hybrid method!")
        print(f"  Dimension: {dimension}D")
        print(f"  Motif measure: {avg_motif_measure:.3f}")
        print(f"  Cell measure: {unit_cell_measure:.3f}")
        if dimension == 2:
            print(f"  Method used: {method_used}")
            print(f"  Parallel area: {parallel_area:.3f}")
            print(f"  Median area: {median_area:.3f}")
        print(f"  → Setting to None")
        ratio = None
    
    # compute the comparable measure
    comparable = compute_comparable_metrics(ratio, dimension)

    data_list.append({
        "filename": contcar_file,
        "dimension": dimension,
        "avg_motif_measure": avg_motif_measure,
        "unit_cell_volume": unit_cell_volume,
        "unit_cell_measure_used": unit_cell_measure,
        "method_used": method_used,
        "best_face_type": best_face_type,
        "parallel_angle": parallel_angle,
        "parallel_area": parallel_area,
        "median_area": median_area,
        "safety_triggered": safety_triggered,
        "packing_fraction": comparable["packing_fraction"],
        "characteristic_length_ratio": comparable["characteristic_length_ratio"],
    })

print(f"\nProcessing complete. Successfully analyzed {len(data_list)} structures.")
print(f"Safety triggered: {safety_triggered_count} times")

# build the DataFrame
VOL_df = pd.DataFrame(data_list)

# print basic statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

for dim in [2, 3]:
    subset = VOL_df[VOL_df['dimension'] == dim]
    if len(subset) > 0:
        print(f"\n{dim}D Structures (n={len(subset)}):")
        if dim == 2:
            print(f"  Method: HYBRID (parallel + median safeguard)")
            
            # distribution of methods used
            method_counts = subset['method_used'].value_counts()
            for method, count in method_counts.items():
                pct = count / len(subset) * 100
                print(f"    {method}: {count} ({pct:.1f}%)")
            
            if subset['parallel_angle'].notna().sum() > 0:
                print(f"  Avg alignment angle: {subset['parallel_angle'].mean():.1f}°")
        else:
            print(f"  Method: Volume / Volume")
        
        if subset['packing_fraction'].notna().sum() > 0:
            print(f"  Packing fraction:")
            print(f"    Mean: {subset['packing_fraction'].mean():.4f}")
            print(f"    Std:  {subset['packing_fraction'].std():.4f}")
            print(f"    Min:  {subset['packing_fraction'].min():.4f}")
            print(f"    Max:  {subset['packing_fraction'].max():.4f}")
            
            # check for values above 1
            over_one = subset[subset['packing_fraction'] > 1.0]
            if len(over_one) > 0:
                print(f"    ⚠️  f>1: {len(over_one)} structures ({len(over_one)/len(subset)*100:.1f}%)")
        
        if subset['characteristic_length_ratio'].notna().sum() > 0:
            print(f"  Characteristic length ratio:")
            print(f"    Mean: {subset['characteristic_length_ratio'].mean():.4f}")
            print(f"    Std:  {subset['characteristic_length_ratio'].std():.4f}")

# write the CSV
output_file = "motif_metrics_hybrid.csv"
VOL_df.to_csv(output_file, index=False)
print(f"\n✅ Results saved to: {output_file}")

print("\n" + "="*80)
print("⭐ HYBRID METHOD SUMMARY ⭐")
print(f"  Safety factor: {SAFETY_FACTOR}")
print(f"  Safety triggered: {safety_triggered_count}/{len(VOL_df[VOL_df['dimension']==2])} 2D structures")
print("  Advantage: Best balance of accuracy and stability")
print("="*80 + "\n")

VOL_df

In [ ]:
VOL_df[VOL_df['characteristic_length_ratio']>0.7]

In [ ]:
VOL_df.describe()

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(VOL_df['characteristic_length_ratio'], VOL_df['packing_fraction'], c=VOL_df['dimension'])

In [ ]:
rows_with_nan = VOL_df[VOL_df.isna().any(axis=1)]
rows_with_nan

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pymatgen.core import Structure
from pymatgen.analysis.local_env import CrystalNN
from scipy.spatial.distance import pdist

# ===== adjust for your environment =====
# list of magnetic elements (example); adjust for your project
magnetic_species = {"Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Y", "Zr", "Nb", "Mo", "Ru", "Rh", "Pd", "Ag", "Cd"}
# file pattern: change to "CONTCAR*" or a path-qualified pattern if needed
FILE_PATTERN = os.path.join("", "POSCAR_*")
# maximum number of ligand neighbours to select per motif
NUM_NEIGHBORS = 12
# ==================================

def find_motifs(contcar_file, magnetic_species, num_neighbors):
    """
    Extract the two magnetic-atom-centred motifs from a POSCAR/CONTCAR file.
    
    Each motif contains:
      - center_coord: coordinates of the central magnetic atom (absolute)
      - motif_coords: ligand positions relative to the central magnetic atom (centred)
      - selected_neighbor_indices: original indices of the selected ligands
      - distances: distance from the centre to each neighbour
    """
    structure = Structure.from_file(contcar_file)
    
    # extract the magnetic-atom indices
    magnetic_indices = [i for i, site in enumerate(structure) if str(site.specie) in magnetic_species]
    if len(magnetic_indices) < 2:
        raise ValueError("fewer than two magnetic atoms in the file")
    
    # if more than two magnetic atoms are present, use only the first two
    magnetic_indices = magnetic_indices[:2]
    
    cnn = CrystalNN()
    motifs = {}
    
    for mag_index in magnetic_indices:
        center_site = structure[mag_index]
        nn_info = cnn.get_nn_info(structure, mag_index)
        
        # keep only ligand neighbours
        nonmag_neighbors = []
        nonmag_neighbor_indices = []
        for info in nn_info:
            neighbor_index = info.get("site_index", None)
            neighbor_site = info["site"]
            if str(neighbor_site.specie) not in magnetic_species:
                nonmag_neighbors.append(neighbor_site.coords)
                nonmag_neighbor_indices.append(neighbor_index)
        
        nonmag_neighbors = np.array(nonmag_neighbors, dtype=float)
        
        if nonmag_neighbors.shape[0] == 0:
            raise ValueError(f"no ligand neighbour found around magnetic atom index {mag_index}")
        
        # select the nearest neighbours by distance from the centre
        distances = np.linalg.norm(nonmag_neighbors - center_site.coords, axis=1)
        actual_neighbors = min(num_neighbors, nonmag_neighbors.shape[0])
        sorted_idx = np.argsort(distances)[:actual_neighbors]
        selected_neighbors = nonmag_neighbors[sorted_idx]
        selected_indices = [nonmag_neighbor_indices[i] for i in sorted_idx]
        selected_distances = distances[sorted_idx]
        
        # subtract the centre to obtain coordinates relative to the magnetic atom
        motif_coords = selected_neighbors - center_site.coords
        
        motifs[mag_index] = {
            "center_coord": center_site.coords,
            "motif_coords": motif_coords,
            "selected_neighbor_indices": selected_indices,
            "distances": selected_distances
        }
    
    return motifs

def sort_motif_by_neighbor_indices(motif):
    """
    Sort the motif in ascending order of 'selected_neighbor_indices' and
    reorder 'motif_coords' and 'distances' to match.
    """
    sel_idx = np.array(motif["selected_neighbor_indices"])
    order = np.argsort(sel_idx)
    motif["selected_neighbor_indices"] = sel_idx[order].tolist()
    motif["motif_coords"] = motif["motif_coords"][order]
    motif["distances"] = motif["distances"][order]
    return motif

def compute_motif_axis_features(motif):
    """
    Compute all pairwise distances between the ligands of the motif and return
    the longest (long axis), the shortest (short axis) and their ratio.
    Returns None if fewer than two points are present.
    """
    points = motif["motif_coords"]
    if points.shape[0] < 2:
        return {"long_axis": None, "short_axis": None, "axis_ratio": None}
    
    pairwise_distances = pdist(points, metric='euclidean')
    long_axis = float(pairwise_distances.max())
    short_axis = float(pairwise_distances.min())
    axis_ratio = (long_axis / short_axis) if short_axis != 0 else np.nan
    
    return {"long_axis": long_axis, "short_axis": short_axis, "axis_ratio": axis_ratio}

def compute_motif_s_delta(motif, eps=1e-12):
    """
    Compute the shape-distortion magnitude s and the off-centring magnitude delta for a motif.
    
    Definitions:
      - C = the mean (centroid) of the ligand positions
      - r_i = distance of each ligand from C
      - rbar = the mean of r_i
      - s = std(r_i) / rbar
      - delta = || r_M - C || / rbar
        (motif_coords are relative to the centre, so delta = ||mean(motif_coords)|| / rbar)
    """
    P = np.asarray(motif["motif_coords"], dtype=float)  # (N,3) coordinates relative to the centre
    if P.shape[0] < 2:
        return {"s": None, "delta": None, "rbar": None, "n_neighbors": int(P.shape[0])}

    # mean ligand vector in the relative frame
    mu = P.mean(axis=0)  # (3,)

    # radii from the ligand centroid (computable from the relative coordinates alone)
    r = np.linalg.norm(P - mu, axis=1)  # (N,)
    rbar = float(r.mean())
    if rbar < eps:
        return {"s": np.nan, "delta": np.nan, "rbar": rbar, "n_neighbors": int(P.shape[0])}

    s = float(r.std(ddof=0) / rbar)
    delta = float(np.linalg.norm(mu) / rbar)

    return {"s": s, "delta": delta, "rbar": rbar, "n_neighbors": int(P.shape[0])}

# ===================== main loop =====================
data_list = []

# collect the file list
contcar_files = glob.glob(FILE_PATTERN)
file_list = [f.replace(os.sep, "/") for f in contcar_files]

for contcar_file in file_list:
    try:
        motifs = find_motifs(contcar_file, magnetic_species=magnetic_species, num_neighbors=NUM_NEIGHBORS)
    except Exception:
        continue  # skip on error
    
    keys = list(motifs.keys())
    if len(keys) < 2:
        continue  # skip if fewer than two magnetic atoms
    
    # sort the motifs of the first two magnetic atoms
    motif0 = sort_motif_by_neighbor_indices(motifs[keys[0]])
    motif1 = sort_motif_by_neighbor_indices(motifs[keys[1]])
    
    # axis features
    axis_features0 = compute_motif_axis_features(motif0)
    axis_features1 = compute_motif_axis_features(motif1)
    
    combined_axis = {}
    if axis_features0["long_axis"] is not None and axis_features1["long_axis"] is not None:
        combined_axis["avg_long_axis"] = (axis_features0["long_axis"] + axis_features1["long_axis"]) / 2.0
    else:
        combined_axis["avg_long_axis"] = None
    if axis_features0["short_axis"] is not None and axis_features1["short_axis"] is not None:
        combined_axis["avg_short_axis"] = (axis_features0["short_axis"] + axis_features1["short_axis"]) / 2.0
    else:
        combined_axis["avg_short_axis"] = None
    if axis_features0["axis_ratio"] is not None and axis_features1["axis_ratio"] is not None:
        combined_axis["avg_axis_ratio"] = (axis_features0["axis_ratio"] + axis_features1["axis_ratio"]) / 2.0
    else:
        combined_axis["avg_axis_ratio"] = None

    # compute s and delta for each of the two magnetic atoms
    sd0 = compute_motif_s_delta(motif0)
    sd1 = compute_motif_s_delta(motif1)

    combined_sd = {}
    if (sd0["s"] is not None and sd1["s"] is not None):
        combined_sd["avg_s"] = (sd0["s"] + sd1["s"]) / 2.0
    else:
        combined_sd["avg_s"] = None

    if (sd0["delta"] is not None and sd1["delta"] is not None):
        combined_sd["avg_delta"] = (sd0["delta"] + sd1["delta"]) / 2.0
    else:
        combined_sd["avg_delta"] = None

    # optional: uncomment to report the individual values as well
    # combined_sd.update({
    #     "s_motif0": sd0["s"], "delta_motif0": sd0["delta"],
    #     "s_motif1": sd1["s"], "delta_motif1": sd1["delta"]
    # })

    file_data = {"filename": contcar_file}
    file_data.update(combined_axis)
    file_data.update(combined_sd)
    data_list.append(file_data)

# build the DataFrame
ELONG_df = pd.DataFrame(data_list)
ELONG_df

In [ ]:
rows_with_nan = ELONG_df[ELONG_df.isna().any(axis=1)]
rows_with_nan

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pymatgen.core import Structure
from pymatgen.analysis.local_env import CrystalNN

def find_motifs(contcar_file, magnetic_species, num_neighbors):
    """
    Extract the two magnetic-atom-centred motifs from a CONTCAR file.
    
    Each motif contains:
      - center_coord: coordinates of the central magnetic atom
      - magnetic_atom_info: atomic number and electronegativity of the central magnetic atom
      - motif_coords: positions of the selected ligands relative to the central atom
      - selected_neighbor_indices: original indices of the selected ligands
      - distances: distance from the centre to each neighbour
      - nonmagnetic_atoms_info: atomic number and electronegativity of each selected ligand
      - num_nonmagnetic_atoms: number of ligands selected in the motif
    """
    structure = Structure.from_file(contcar_file)
    
    # extract the magnetic-atom indices
    magnetic_indices = [i for i, site in enumerate(structure) if str(site.specie) in magnetic_species]
    if len(magnetic_indices) < 2:
        raise ValueError("fewer than two magnetic atoms in the CONTCAR")
    
    # if more than two magnetic atoms are present, use only the first two
    magnetic_indices = magnetic_indices[:2]
    
    cnn = CrystalNN()
    motifs = {}
    
    for mag_index in magnetic_indices:
        center_site = structure[mag_index]
        center_species = center_site.specie
        magnetic_atom_info = {"atomic_number": center_species.Z, "electronegativity": center_species.X}
        
        nn_info = cnn.get_nn_info(structure, mag_index)
        
        # select the ligands (elements absent from the magnetic-species list)
        nonmag_neighbors = []
        nonmag_neighbor_indices = []
        nonmag_neighbors_species = []
        for info in nn_info:
            neighbor_index = info.get("site_index", None)
            neighbor_site = info["site"]
            if str(neighbor_site.specie) not in magnetic_species:
                nonmag_neighbors.append(neighbor_site.coords)
                nonmag_neighbor_indices.append(neighbor_index)
                nonmag_neighbors_species.append(neighbor_site.specie)
        
        nonmag_neighbors = np.array(nonmag_neighbors)
        
        if nonmag_neighbors.shape[0] == 0:
            raise ValueError(f"no ligand found around magnetic atom index {mag_index}")
        
        # select the nearest neighbours by distance from the centre
        distances = np.linalg.norm(nonmag_neighbors - center_site.coords, axis=1)
        actual_neighbors = min(num_neighbors, nonmag_neighbors.shape[0])
        sorted_idx = np.argsort(distances)[:actual_neighbors]
        selected_neighbors = nonmag_neighbors[sorted_idx]
        selected_indices = [nonmag_neighbor_indices[i] for i in sorted_idx]
        selected_distances = distances[sorted_idx]
        selected_neighbors_species = [nonmag_neighbors_species[i] for i in sorted_idx]
        # a single ligand species is assumed, so the first element's data are used
        nonmag_atoms_info = [{"atomic_number": selected_neighbors_species[0].Z,
                              "electronegativity": selected_neighbors_species[0].X}]
        
        # subtract the centre to obtain centred relative coordinates (used where needed)
        motif_coords = selected_neighbors - center_site.coords
        
        num_nonmagnetic_atoms = len(selected_indices)
        
        motifs[mag_index] = {
            "center_coord": center_site.coords,
            "magnetic_atom_info": magnetic_atom_info,
            "motif_coords": motif_coords,
            "selected_neighbor_indices": selected_indices,
            "distances": selected_distances,
            "nonmagnetic_atoms_info": nonmag_atoms_info,
            "num_nonmagnetic_atoms": num_nonmagnetic_atoms
        }
    
    return motifs

def sort_motif_by_neighbor_indices(motif):
    """
    Sort the motif in ascending order of 'selected_neighbor_indices' and
    reorder 'motif_coords', 'distances' and 'nonmagnetic_atoms_info' to match.
    """
    sel_idx = np.array(motif["selected_neighbor_indices"])
    order = np.argsort(sel_idx)
    motif["selected_neighbor_indices"] = sel_idx[order].tolist()
    motif["motif_coords"] = motif["motif_coords"][order]
    motif["distances"] = motif["distances"][order]
    # a single ligand species is present, so reordering can be skipped
    return motif

# build the DataFrame by collecting the per-file results
data_list = []

# list of CONTCAR files (adjust the filename pattern for your setup)
contcar_files = glob.glob(os.path.join("", "POSCAR_*"))
file_list = [f.replace(os.sep, "/") for f in contcar_files]

for contcar_file in file_list:
    try:
        motifs = find_motifs(contcar_file, magnetic_species=magnetic_species, num_neighbors=12)
    except Exception:
        continue  # skip this file on error
    
    keys = list(motifs.keys())
    if len(keys) < 2:
        continue  # skip if fewer than two magnetic atoms
    
    # sort the motif data of the two magnetic atoms
    motif0 = sort_motif_by_neighbor_indices(motifs[keys[0]])
    motif1 = sort_motif_by_neighbor_indices(motifs[keys[1]])
    
    # the magnetic-atom data are assumed identical for both motifs
    mag_info = motif0["magnetic_atom_info"]
    # ligand data: a single species is assumed, so the first ligand of motif 0 is used
    nonmag_info = motif0["nonmagnetic_atoms_info"][0]
    
    # number of ligands in each motif
    count_motif0 = motif0["num_nonmagnetic_atoms"]
    count_motif1 = motif1["num_nonmagnetic_atoms"]
    
    # collect one row per file
    file_data = {
        "filename": contcar_file,
        "motif0_nonmag_count": count_motif0,
        "motif1_nonmag_count": count_motif1,
        "magnetic_atomic_number": mag_info["atomic_number"],
        "magnetic_electronegativity": mag_info["electronegativity"],
        "nonmagnetic_atomic_number": nonmag_info["atomic_number"],
        "nonmagnetic_electronegativity": nonmag_info["electronegativity"]
    }
    data_list.append(file_data)

# build the DataFrame (stored in df, not printed)
ATOM_df = pd.DataFrame(data_list)
ATOM_df

In [ ]:
ATOM_df

In [ ]:
rows_with_nan = ATOM_df[ATOM_df.isna().any(axis=1)]
rows_with_nan

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pymatgen.core import Structure
from pymatgen.analysis.local_env import CrystalNN
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import pdist, cdist  # cdist added, for the Hungarian matching

# (the earlier function definitions are unchanged and omitted)

def _relative_vector_with_pbc(structure, center_index, nn_info):
    """
    Compute the minimum-image centre-to-neighbour vector (in Angstrom) from a CrystalNN.get_nn_info() entry.
    """
    jimg = nn_info.get("image", (0, 0, 0))
    if jimg is None:
        jimg = (0, 0, 0)
    jimg = np.array(jimg, dtype=int)
    nbr_idx = nn_info.get("site_index", None)
    if nbr_idx is None:
        return None

    f_center = structure[center_index].frac_coords
    f_nbr = structure[nbr_idx].frac_coords + jimg
    f_vec = f_nbr - f_center
    # convert the fractional vector to Cartesian
    rel_cart = structure.lattice.get_cartesian_coords(f_vec)
    return rel_cart

def find_motifs(contcar_file, magnetic_species=None, r_cut=3.0, num_neighbors=12):
    """
    Extract the two magnetic-atom-centred motifs from a CONTCAR file.
    (modified so that the centre-to-neighbour vectors respect periodic boundaries)
    """
    structure = Structure.from_file(contcar_file)
    
    # find the site indices of the magnetic atoms
    magnetic_indices = [i for i, site in enumerate(structure) if str(site.specie) in magnetic_species]
    if len(magnetic_indices) < 2:
        raise ValueError("fewer than two magnetic atoms in the CONTCAR")
    
    # if more than two magnetic atoms are present, use only the first two
    magnetic_indices = magnetic_indices[:2]
    
    cnn = CrystalNN()
    motifs = {}
    
    for mag_index in magnetic_indices:
        center_site = structure[mag_index]
        nn_info_list = cnn.get_nn_info(structure, mag_index)
        
        # keep only ligands (elements absent from the magnetic-species list)
        rel_vectors = []               # centre-relative vectors in Angstrom, minimum image under PBC
        nonmag_neighbor_indices = []   # site index of the corresponding neighbour
        distances = []                 # norm of the relative vector, i.e. the distance
        
        for info in nn_info_list:
            neighbor_index = info.get("site_index", None)
            if neighbor_index is None:
                continue
            
            neighbor_site = structure[neighbor_index]
            if str(neighbor_site.specie) in magnetic_species:
                continue  # ligand neighbours only
            
            # === apply PBC: compute the minimum-image relative vector ===
            rel_vec = _relative_vector_with_pbc(structure, mag_index, info)
            if rel_vec is None:
                continue
            rel_vectors.append(rel_vec)
            nonmag_neighbor_indices.append(neighbor_index)
            distances.append(np.linalg.norm(rel_vec))
        
        if len(rel_vectors) == 0:
            raise ValueError(f"no ligand found around magnetic atom index {mag_index}")
        
        rel_vectors = np.array(rel_vectors)
        distances = np.array(distances)
        
        # select the nearest neighbours by distance from the centre
        actual_neighbors = min(num_neighbors, rel_vectors.shape[0])
        sorted_idx = np.argsort(distances)[:actual_neighbors]
        
        selected_rel_vectors = rel_vectors[sorted_idx]               # already centred relative coordinates
        selected_indices = [nonmag_neighbor_indices[i] for i in sorted_idx]
        selected_distances = distances[sorted_idx]
        
        # motif_coords are already centred relative coordinates, in Angstrom
        motif_coords = selected_rel_vectors
        
        motifs[mag_index] = {
            "center_coord": center_site.coords,
            "motif_coords": motif_coords,
            "selected_neighbor_indices": selected_indices,
            "distances": selected_distances
        }
    
    return motifs

def sort_motif_by_neighbor_indices(motif):
    """
    Sort the motif dict in ascending order of 'selected_neighbor_indices' and
    reorder 'motif_coords' and 'distances' to match.
    (changed to return a copy rather than modify the original)
    """
    sel_idx = np.array(motif["selected_neighbor_indices"])
    order = np.argsort(sel_idx)

    # return a copy, with no side effects
    return {
        "center_coord": np.array(motif["center_coord"]).copy(),
        "motif_coords": motif["motif_coords"][order].copy(),
        "selected_neighbor_indices": sel_idx[order].tolist(),
        "distances": motif["distances"][order].copy()
    }

def kabsch_rotation(P, Q):
    """
    Apply the Kabsch algorithm to two point sets P and Q, each of shape (N, 3), to obtain the optimal rotation matrix R.
    (the body is unchanged from the original)
    """
    C = np.dot(P.T, Q)  # covariance matrix
    V, S, Wt = np.linalg.svd(C)
    d = np.linalg.det(np.dot(Wt.T, V.T))
    D = np.eye(3)
    D[2, 2] = d
    R = np.dot(np.dot(Wt.T, D), V.T)
    return R

def rotation_angle_from_matrix(R):
    """
    Compute the rotation angle theta from the 3x3 rotation matrix R.
    (the body is unchanged from the original)
    """
    trace = np.trace(R)
    cos_theta = np.clip((trace - 1) / 2, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    return theta

# === robust rotation angle based on Hungarian matching ===
def rotation_angle_with_assignment(P, Q, max_iter=8, tol=1e-10):
    """
    P, Q: (N, 3) centre-relative coordinates, in Angstrom.
    1) initial Kabsch fit
    2) rotate P, then match against Q by Hungarian assignment on the squared-distance cost matrix
    3) update the Kabsch fit on the matched pairs
    Iterate to convergence and return the final rotation angle in degrees.
    """
    n = min(len(P), len(Q))
    if n < 3:
        raise ValueError("at least three neighbours are required for a stable rotation")
    P0 = np.asarray(P[:n], float)
    Q0 = np.asarray(Q[:n], float)

    # initial rotation estimate
    R = kabsch_rotation(P0, Q0)
    for _ in range(max_iter):
        # rotate P and build the cost matrix against Q (squared Euclidean)
        P_rot = P0 @ R
        C = cdist(P_rot, Q0, metric="sqeuclidean")
        row, col = linear_sum_assignment(C)

        R_new = kabsch_rotation(P0[row], Q0[col])
        if np.linalg.norm(R_new - R) < tol:
            R = R_new
            break
        R = R_new

    theta_deg = np.degrees(rotation_angle_from_matrix(R))
    return theta_deg

# build the DataFrame by collecting the rotation angle of each file
results = []

# list of CONTCAR files (adjust for your setup)
contcar_files = glob.glob(os.path.join("", "POSCAR_*"))
file_list = [f.replace(os.sep, "/") for f in contcar_files]

for contcar_file in file_list:
    try:
        motifs = find_motifs(
            contcar_file,
            magnetic_species=["Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", "Y", "Zr", "Nb", "Mo", "Ru", "Rh", "Pd", "Ag", "Cd"],
            num_neighbors=12
        )
        keys = list(motifs.keys())
        if len(keys) < 2:
            continue
        
        motif0_orig = motifs[keys[0]]
        motif1_orig = motifs[keys[1]]
        
        # --- make the two motifs the same size, as the Kabsch algorithm requires ---
        min_len = min(len(motif0_orig['motif_coords']), len(motif1_orig['motif_coords']))
        
        # === skip motifs with fewer than three neighbours ===
        if min_len < 3:
            continue  # the rotation is ill-defined; skip this file
        
        # ======================= start of the modified section =======================
        # 1. index-independent calculation (sorted by distance, comparing geometric shape)
        # the output of find_motifs is already sorted by distance and is used as is
        coords0_dist = motif0_orig['motif_coords'][:min_len]
        coords1_dist = motif1_orig['motif_coords'][:min_len]

        # 3. robust rotation angle from Hungarian matching
        angle_deg_hungarian = rotation_angle_with_assignment(coords0_dist, coords1_dist)
        
        # record all three angle definitions in the result
        results.append({
            "filename": contcar_file,
            "hungarian_rotation_angle_deg": angle_deg_hungarian
        })
        # ======================= end of the modified section =========================
        
    except Exception:
        continue

# build the DataFrame (stored in df, not printed)
ROT_df = pd.DataFrame(results)
ROT_df

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pymatgen.core import Structure
from pymatgen.analysis.local_env import CrystalNN
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

def find_motifs(contcar_file, magnetic_species=None, num_neighbors=12):
    """
    Extract the two magnetic-atom-centred motifs from a CONTCAR file.
    (periodic boundaries are handled automatically)
    """
    structure = Structure.from_file(contcar_file)
    if magnetic_species is None:
        magnetic_species = ["Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn", 
                            "Y", "Zr", "Nb", "Mo", "Ru", "Rh", "Pd", "Ag", "Cd"]
    
    magnetic_indices = [i for i, site in enumerate(structure) if str(site.specie) in magnetic_species]
    if len(magnetic_indices) < 2:
        raise ValueError("fewer than two magnetic atoms in the file")
    
    magnetic_indices = magnetic_indices[:2]
    
    cnn = CrystalNN()
    motifs = {}
    
    for mag_index in magnetic_indices:
        center_site = structure[mag_index]
        nn_info = cnn.get_nn_info(structure, mag_index)
        
        nonmag_neighbors_coords = np.array([
            info["site"].coords for info in nn_info 
            if str(info["site"].specie) not in magnetic_species
        ])
        
        if nonmag_neighbors_coords.shape[0] == 0:
            raise ValueError(f"no ligand neighbour found around magnetic atom index {mag_index}")
        
        distances = np.linalg.norm(nonmag_neighbors_coords - center_site.coords, axis=1)
        actual_neighbors = min(num_neighbors, nonmag_neighbors_coords.shape[0])
        sorted_indices = np.argsort(distances)[:actual_neighbors]
        
        selected_neighbors_coords = nonmag_neighbors_coords[sorted_indices]
        motif_coords = selected_neighbors_coords - center_site.coords
        
        # mean M-X bond length (l0)
        selected_distances = distances[sorted_indices]
        avg_bond_length = np.mean(selected_distances)
        
        motifs[mag_index] = {
            "motif_coords": motif_coords,
            "avg_bond_length": avg_bond_length  # store l0
        }
    
    return motifs

def calculate_rmsd_and_std_with_assignment(p_coords, q_coords):
    """
    Compute the optimal RMSD between two motifs P and Q and the standard deviation of the distances.
    (compared in the given orientation, with no rotational alignment)
    """
    cost_matrix = cdist(p_coords, q_coords, 'sqeuclidean')
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    optimal_squared_dists = cost_matrix[row_ind, col_ind]
    rmsd = np.sqrt(np.mean(optimal_squared_dists))
    
    optimal_dists = np.sqrt(optimal_squared_dists)
    std = np.std(optimal_dists)
    
    return rmsd, std

def calculate_p_breaking_metric(motif1_coords, motif2_coords, l0_motif1, l0_motif2):
    """
    Take the two motif coordinate sets and compute the symmetry-breaking metric and its standard deviation.
    Formula: feat_x = Dbar_p * Dbar_i = (D_p/l0) * (D_i/l0)
    """
    if motif1_coords.shape[0] != motif2_coords.shape[0]:
        raise ValueError("the two motifs contain different numbers of ligands")
        
    # case 1: identity (compared as given)
    rmsd_identity, std_identity = calculate_rmsd_and_std_with_assignment(motif1_coords, motif2_coords)
    
    # case 2: inversion (compared after point inversion)
    motif1_coords_inverted = -motif1_coords
    rmsd_inversion, std_inversion = calculate_rmsd_and_std_with_assignment(motif1_coords_inverted, motif2_coords)

    # normalise by the mean bond length
    # D̄_p = D_p / ℓ₀, D̄_i = D_i / ℓ₀
    avg_l0 = (l0_motif1 + l0_motif2) / 2  # use the mean l0 of the two motifs
    
    normalized_rmsd_identity = rmsd_identity / avg_l0
    normalized_rmsd_inversion = rmsd_inversion / avg_l0
    
    # P-breaking metric: D̄_p × D̄_i
    p_metric = normalized_rmsd_identity * normalized_rmsd_inversion
    
    if rmsd_identity <= rmsd_inversion:
        p_metric_std = std_identity / avg_l0
        is_inversion_symmetric = 0
    else:
        p_metric_std = std_inversion / avg_l0
        is_inversion_symmetric = 1

    return p_metric, p_metric_std, is_inversion_symmetric

# ==============================================================================
# main driver
# ==============================================================================
def main():
    search_path = ""
    file_pattern = "POSCAR_*"
    results = []

    contcar_files = glob.glob(os.path.join(search_path, file_pattern))
    file_list = [f.replace(os.sep, "/") for f in contcar_files]
    print(f"analysing {len(file_list)} files...")

    for contcar_file in file_list:
        try:
            motifs = find_motifs(contcar_file, num_neighbors=12)
            keys = list(motifs.keys())
            
            motif0_coords = motifs[keys[0]]["motif_coords"]
            motif1_coords = motifs[keys[1]]["motif_coords"]
            l0_motif0 = motifs[keys[0]]["avg_bond_length"]
            l0_motif1 = motifs[keys[1]]["avg_bond_length"]
            
            p_metric, p_metric_std, is_inv = calculate_p_breaking_metric(
                motif0_coords, motif1_coords, l0_motif0, l0_motif1
            )
            
            results.append({
                "filename": contcar_file,
                "p_metric": p_metric,
                "p_metric_std": p_metric_std,
                "avg_bond_length_motif0": l0_motif0,
                "avg_bond_length_motif1": l0_motif1
            })
        except Exception as e:
            print(f" (!) error while processing {contcar_file}: {e}")
            continue

    if results:
        df = pd.DataFrame(results)
        print("\nanalysis complete; the DataFrame has been created.")
        return df
    else:
        print("\nno files to process, or every file raised an error.")
        return pd.DataFrame()

if __name__ == "__main__":
    P_METRIC_df = main()
    if not P_METRIC_df.empty:
        print("\n--- preview of the DataFrame ---")
P_METRIC_df

In [ ]:
P_METRIC_df.describe()

In [ ]:
P_METRIC_df[P_METRIC_df['p_metric']<0.001]

In [ ]:
Hubbard_U = {"21": 3, "22": 3, "23": 4, "24": 4 , "25": 4, "26": 4, "27": 3, "28": 7, "29": 7, "30": 7, "39": 3, "40": 3, "41": 4, "42": 4, "44": 4, "45": 4, "46": 4, "47": 0, "48": 0}
d_orbit_e = {"21": 2, "22": 3, "23": 4, "24": 5 , "25": 6, "26": 7, "27": 8, "28": 9, "29": 10, "30": 10, "39": 2, "40": 3, "41": 4, "42": 5, "44": 7, "45": 8, "46": 9, "47": 10, "48": 10}

non_mag_p_orbit_e = {"5": 1, "6": 2, "7": 3, "8": 4, "9": 5, "13": 1, "14": 2, "15": 3, "16": 4, "17": 5, "31": 1, "32": 2, "33": 3, "34": 4, "35": 5, "49": 1,"50": 2, "51": 3, "52": 4, "53": 5, "81": 1, "82": 2, "83": 3}

ZVAL = {"21": 11, "22": 12, "23": 13, "24": 12 , "25": 13, "26": 8, "27": 9, "28": 10, "29": 11, "30": 12, "39": 11, "40": 12, "41": 13, "42": 14, "44": 14, "45": 15, "46": 10, "47": 11, "48": 12, "5": 3, "6": 4, "7": 5, "8": 6, "9": 7, "13": 3, "14": 4, "15": 5, "16": 6, "17": 7, "31": 13, "32": 14, "33": 5, "34": 6, "35": 7, "49": 13,"50": 14, "51": 5, "52": 6, "53": 7, "81": 13, "82": 14, "83": 15}

d_lone_pair = {"21": 1, "22": 2, "23": 3, "24": 5, "25": 5, "26": 4, "27": 3, "28": 2, "29": 0, "30": 0, "39": 1, "40": 2, "41": 4, "42": 5, "44": 3, "45": 2, "46": 0, "47": 0, "48": 0}

proxy_map = {"21":0, "22":1.73, "23":2.83, "24": 3.87, "25": 4.90, "26": 4.90, "27": 3.87, "28": 2.83, "29": 1.73, "30": 0, "39":0, "40":1.73, "41": 2.83, "42": 3.87, "44": 4.90, "45": 3.87, "46": 2.83, "47": 1.73, "48":0}


In [ ]:
VOL_df.columns

In [ ]:
import os
# the data location is set by an environment variable; it defaults to the current directory
#   e.g. export ALTERMAG_DATA=/path/to/Altermagnetism
DATA_DIR = os.environ.get('ALTERMAG_DATA', '.')
OUT_DIR  = os.environ.get('ALTERMAG_OUT', DATA_DIR)
merge_df = BOND_df.merge(NN_df, on='filename', how='inner')
merge_df = merge_df.merge(ELONG_df, on='filename', how='inner')
merge_df = merge_df.merge(ATOM_df, on='filename', how='inner')
merge_df = merge_df.merge(ROT_df, on='filename', how='inner')
merge_df = merge_df.merge(VOL_df, on='filename', how='inner')
merge_df = merge_df.merge(P_METRIC_df, on='filename', how='inner')

merge_df["d_orb_e"] = merge_df["magnetic_atomic_number"].astype(str).map(d_orbit_e)
merge_df["p_orb_e_non"] = merge_df["nonmagnetic_atomic_number"].astype(str).map(non_mag_p_orbit_e)
merge_df["d_lone_pair"] = merge_df['magnetic_atomic_number'].astype(str).map(d_lone_pair)
merge_df["proxy_M_magnet"] = merge_df['magnetic_atomic_number'].astype(str).map(proxy_map)

merge_df['delta_chi'] = merge_df['magnetic_electronegativity'] - merge_df['nonmagnetic_electronegativity']
merge_df['abs_delta_chi'] = merge_df['delta_chi'].abs()

merge_df['delta_Z'] = merge_df['magnetic_atomic_number'] - merge_df['nonmagnetic_atomic_number']
merge_df['abs_delta_Z'] = merge_df['delta_Z'].abs()

merge_df['pd_ratio'] = merge_df['p_orb_e_non'] / (merge_df['d_orb_e'] + 1e-6)

merge_df['ax_eq_gap']  = merge_df['max_bond_length'] - merge_df['avg_bond_length']
merge_df['bond_range'] = merge_df['max_bond_length'] - merge_df['min_bond_length']
merge_df['bond_cv']    = merge_df['std_bond_length'] / (merge_df['avg_bond_length'] + 1e-6)

merge_df['center_angle_spread'] = merge_df['center_max_angle'] - merge_df['center_min_angle']
merge_df['nonmag_angle_spread'] = merge_df['nonmag_max_angle'] - merge_df['nonmag_min_angle']

merge_df['delta_chi_times_axeq'] = merge_df['abs_delta_chi'] * merge_df['ax_eq_gap']

for k in ['1st','2nd','3rd']:
    merge_df[f'd_global_local_{k}'] = merge_df[f'global_{k}'] - merge_df[f'labelled_{k}']


AL_df = pd.read_csv(os.path.join(DATA_DIR, 'Altermagnetism_full_data.csv'))
merge_df = merge_df.merge(AL_df, on='filename', how='inner')
merge_df.columns

In [ ]:
for aaa in list(merge_df.columns):
    print("'" + aaa + "',")

In [ ]:
import os
# the data location is set by an environment variable; it defaults to the current directory
#   e.g. export ALTERMAG_DATA=/path/to/Altermagnetism
DATA_DIR = os.environ.get('ALTERMAG_DATA', '.')
OUT_DIR  = os.environ.get('ALTERMAG_OUT', DATA_DIR)
merge_df = merge_df[['filename','maximum splitting energy',
'avg_bond_length',
'max_bond_length',
'min_bond_length',
'std_bond_length',
'center_max_angle',
'center_min_angle',
'center_avg_angle',
'center_std_angle',
'nonmag_max_angle',
'nonmag_min_angle',
'nonmag_std_angle',
'labelled_1st',
'labelled_2nd',
'labelled_3rd',
'global_1st',
'global_2nd',
'global_3rd',
'avg_long_axis',
'avg_short_axis',
'avg_axis_ratio',
'avg_s',
'avg_delta',
'motif0_nonmag_count',
'magnetic_atomic_number',
'magnetic_electronegativity',
'nonmagnetic_atomic_number',
'nonmagnetic_electronegativity',
'hungarian_rotation_angle_deg',
'dimension',
'avg_motif_measure',
'unit_cell_volume',
'packing_fraction',
'characteristic_length_ratio',
'p_metric',
'p_metric_std',
'd_orb_e',
'p_orb_e_non',
'd_lone_pair',
'proxy_M_magnet',
'delta_chi',
'abs_delta_chi',
'delta_Z',
'abs_delta_Z',
'pd_ratio',
'ax_eq_gap',
'bond_range',
'bond_cv',
'center_angle_spread',
'nonmag_angle_spread',
'delta_chi_times_axeq',
'd_global_local_1st',
'd_global_local_2nd',
'd_global_local_3rd',
'gamma point average splitting',
'ion1 tot',
'tot_mag']]
merge_df['ion1 tot'] = np.abs(merge_df['ion1 tot'])
merge_df['tot_mag'] = np.abs(merge_df['tot_mag'])
merge_df = merge_df[~merge_df.isna().any(axis=1)]
merge_df.to_csv(os.path.join(OUT_DIR, 'AL_ML_AX_strain_data_251110.csv'), index=False)
merge_df

In [ ]:
import matplotlib.pyplot as plt
df = merge_df

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'
# Define target and top 5 features
target = 'maximum splitting energy'
top_5_features = ['center_std_angle', 'avg_s', 'global_1st', 'center_avg_angle', 'nonmag_std_angle']

# Plot scatter plots of each top feature against SSE
plt.figure(figsize=(8, 10))
for i, feature in enumerate(top_5_features, 1):
    plt.subplot(3, 2, i)
    plt.scatter(df[feature], df[target], alpha=0.5, s= 1)
    plt.xlabel(feature)
    plt.ylabel("Maximum Splitting Energy (eV)")
    plt.title(f"SSE vs {feature}")
    plt.grid(True)

plt.tight_layout()
plt.show()
